# Agentic RFP Evaluation — Pipeline Prototyping Notebook

This notebook exercises every stage of the pipeline **before** wiring it into
the Streamlit app, exactly as required by the project brief:

1. Environment setup & dependency check
2. Synthetic PDF generation & text extraction
3. SQLite schema initialization & seeding
4. Mock LLM evaluation call & schema validation
5. Deterministic math (Absolute Score, Benchmark, Gap, Relative %, PPI)
6. Tie-break logic verification with a synthetic edge case
7. End-to-end batch evaluation run

> All cell outputs below were produced by **actually running this code**, not
> hand-written. Cell 4 uses the deterministic offline mock LLM
> (`evaluator.mock_llm_call`) so the whole notebook runs with **no API key and
> no network access** — swap `provider="mock"` for `provider="gemini"` /
> `"openai"` in `evaluator.evaluate_supplier(...)` once you have a key.


## Cell 1 — Environment setup & dependency check

In [1]:
import importlib
import sys

REQUIRED = ["reportlab", "pypdf", "streamlit", "pydantic"]
OPTIONAL = ["fitz", "google.generativeai", "openai"]  # fitz = PyMuPDF

print(f"Python: {sys.version.split()[0]}\n")

for mod in REQUIRED:
    try:
        importlib.import_module(mod)
        print(f"[required] {mod:<20} OK")
    except ImportError:
        print(f"[required] {mod:<20} MISSING -> pip install -r requirements.txt")

for mod in OPTIONAL:
    try:
        importlib.import_module(mod)
        print(f"[optional] {mod:<20} OK")
    except ImportError:
        print(f"[optional] {mod:<20} not installed (a fallback path will be used automatically)")


Python: 3.12.3

[required] reportlab            OK
[required] pypdf                OK
[required] streamlit            MISSING -> pip install -r requirements.txt
[required] pydantic             MISSING -> pip install -r requirements.txt
[optional] fitz                 not installed (a fallback path will be used automatically)
[optional] google.generativeai  not installed (a fallback path will be used automatically)
[optional] openai               not installed (a fallback path will be used automatically)


## Cell 2 — Synthetic PDF generation & text extraction test

Generates the four fictional supplier proposals with `generate_synthetic_pdfs.py`,
then runs them through the Document Tool (`pdf_extractor.py`), which prefers
PyMuPDF and transparently falls back to `pypdf` if PyMuPDF isn't installed.


In [2]:
import generate_synthetic_pdfs
import pdf_extractor

pdf_paths = generate_synthetic_pdfs.generate_all()
print(f"Generated {len(pdf_paths)} synthetic proposals:")
for p in pdf_paths:
    print(f"  - {p}")

print()
extractions = {}
for p in pdf_paths:
    supplier = p.stem.replace("_", " ")
    result = pdf_extractor.extract_text_from_pdf(p)
    extractions[supplier] = result
    print(f"[{result['backend']}] {supplier:<18} -> {result['char_count']:>5} chars extracted")

print()
print("--- First 300 characters extracted from 'Apex Systems' ---")
print(extractions["Apex Systems"]["text"][:300])


Generated 4 synthetic proposals:
  - /home/claude/agentic_rfp_project/data/proposals/Apex_Systems.pdf
  - /home/claude/agentic_rfp_project/data/proposals/BrightPath_Tech.pdf
  - /home/claude/agentic_rfp_project/data/proposals/NexaWorks.pdf
  - /home/claude/agentic_rfp_project/data/proposals/Orbit_Digital.pdf

[pypdf] Apex Systems       ->  2838 chars extracted
[pypdf] BrightPath Tech    ->  2389 chars extracted
[pypdf] NexaWorks          ->  2622 chars extracted
[pypdf] Orbit Digital      ->  2346 chars extracted

--- First 300 characters extracted from 'Apex Systems' ---
Apex Systems
RFP Response — Procurement Evaluation Platform
Submission Date
2026-03-01
Historical Experience Rating
4.8 / 5.0
Prepared for
Procurement Evaluation Committee (synthetic classroom exercise)
Executive Summary
Apex Systems proposes a cloud-native procurement automation platform built on a


## Cell 3 — SQLite schema initialization & seeding

Creates `evaluation_criteria`, `rfp_runs`, and `supplier_results`, then seeds
the five default criteria (idempotent).


In [3]:
import database

database.init_db()
database.seed_default_criteria()

active_criteria = database.get_active_criteria()
print(f"Database file: {database.get_db_path()}")
print(f"Active criteria total weight: {database.total_active_weight()}%\n")
for c in active_criteria:
    print(f"  [{c['criterion_id']}] {c['name']:<24} weight={c['weight']:>5.1f}%  max_score={c['max_score']}")


Database file: /home/claude/agentic_rfp_project/data/rfp_evaluation.db
Active criteria total weight: 100.0%

  [1] Technical Capability     weight= 30.0%  max_score=10.0
  [2] Implementation Plan      weight= 20.0%  max_score=10.0
  [3] Commercial Value         weight= 20.0%  max_score=10.0
  [4] Security & Compliance    weight= 20.0%  max_score=10.0
  [5] Support & Experience     weight= 10.0%  max_score=10.0


## Cell 4 — Mock LLM evaluation call & schema validation

Calls the deterministic offline mock LLM for one supplier, then deliberately
feeds a **malformed** response (out-of-range score + a missing criterion)
through the Validation Tool to prove auto-fill, clipping, and warning
generation all work.


In [4]:
import json
import evaluator
import validator

supplier = "Apex Systems"
proposal_text = extractions[supplier]["text"]

raw_llm_json = evaluator.evaluate_supplier(supplier, proposal_text, active_criteria, provider="mock")
print("--- Raw (mock) LLM JSON ---")
print(json.dumps(raw_llm_json, indent=2)[:900], "...\n")

validated = validator.validate_llm_result(raw_llm_json, active_criteria, expected_supplier_name=supplier)
print(f"Validated {len(validated.criteria)} criteria for {validated.supplier_name}, "
      f"{len(validated.warnings)} warning(s) on a clean mock response.\n")

print("--- Now testing the validator against a deliberately BROKEN LLM response ---")
broken_response = {
    "supplier_name": "Apex Systems",
    "criteria": [
        {"criterion_id": 1, "score": 999, "max_score": 10, "justification": "way out of range", "evidence": ""},
        # criterion_id 2, 3, 4, 5 are all missing on purpose
    ],
    "risks": ["Some risk"],
    "overall_summary": "Deliberately malformed for testing.",
}
validated_broken = validator.validate_llm_result(broken_response, active_criteria, expected_supplier_name=supplier)
print(f"Warnings generated ({len(validated_broken.warnings)}):")
for w in validated_broken.warnings:
    print(" -", w)


--- Raw (mock) LLM JSON ---
{
  "supplier_name": "Apex Systems",
  "criteria": [
    {
      "criterion_id": 1,
      "score": 8.47,
      "max_score": 10.0,
      "justification": "[MOCK EVALUATOR] Found 12 keyword reference(s) related to 'Technical Capability' in the proposal text.",
      "evidence": "technical depth is the strongest differentiator in this proposal. Proposed Solution Our solution is composed of independently deployable mic"
    },
    {
      "criterion_id": 2,
      "score": 2.14,
      "max_score": 10.0,
      "justification": "[MOCK EVALUATOR] Found 2 keyword reference(s) related to 'Implementation Plan' in the proposal text.",
      "evidence": "implementation and $276,667 per year in platform and support fees. Pricing assumes a single production region plus one DR region, up to 500"
    },
    {
      "criterion_id": 3,
      "score": 9.79,
      "max_score": 10.0,
      "justification": "[ ...

Validated 5 criteria for Apex Systems, 0 warning(s) on a clean moc

## Cell 5 — Deterministic math test (Absolute Score, Benchmark, Gap, Relative %, PPI)

Uses `ranker.py` directly on a tiny hand-built two-supplier example so every
formula's output can be checked by hand.


In [5]:
import ranker

demo_scores_a = {
    1: {"score": 9.0, "weight": 30.0, "max_score": 10.0, "name": "Technical Capability"},
    2: {"score": 6.0, "weight": 20.0, "max_score": 10.0, "name": "Implementation Plan"},
}
demo_scores_b = {
    1: {"score": 6.0, "weight": 30.0, "max_score": 10.0, "name": "Technical Capability"},
    2: {"score": 9.0, "weight": 20.0, "max_score": 10.0, "name": "Implementation Plan"},
}

sc_a = ranker.SupplierScorecard("Supplier A", "2026-03-01", 4.5, demo_scores_a)
sc_b = ranker.SupplierScorecard("Supplier B", "2026-03-02", 4.0, demo_scores_b)

print("Absolute score A:", ranker.compute_absolute_score(demo_scores_a))
print("Absolute score B:", ranker.compute_absolute_score(demo_scores_b))

benchmarks = ranker.compute_benchmarks([sc_a, sc_b])
print("Benchmarks per criterion:", benchmarks)

for cid, bench in benchmarks.items():
    rel_a = ranker.compute_relative_performance(demo_scores_a[cid]["score"], bench)
    rel_b = ranker.compute_relative_performance(demo_scores_b[cid]["score"], bench)
    print(f"  criterion {cid}: relative% A={rel_a}, relative% B={rel_b}")

print()
print("--- Zero-benchmark edge case (all suppliers score 0 on a criterion) ---")
print("relative% when benchmark=0, score=0 ->", ranker.compute_relative_performance(0.0, 0.0))
print("relative% when benchmark=0, score=5 (should not normally happen) ->", ranker.compute_relative_performance(5.0, 0.0))


Absolute score A: 39.0
Absolute score B: 36.0
Benchmarks per criterion: {1: 9.0, 2: 9.0}
  criterion 1: relative% A=100.0, relative% B=66.6667
  criterion 2: relative% A=66.6667, relative% B=100.0

--- Zero-benchmark edge case (all suppliers score 0 on a criterion) ---
relative% when benchmark=0, score=0 -> 100.0
relative% when benchmark=0, score=5 (should not normally happen) -> 0.0


## Cell 6 — Tie-break logic verification with synthetic edge-case data

Four suppliers engineered to collide on PPI, then on submission date, then on
experience rating — proving the full 4-level tie-break chain fires in order:
**PPI (desc) → submission date (asc) → experience rating (desc) → name (asc)**.


In [6]:
edge_case_scores = {
    1: {"score": 8.0, "weight": 50.0, "max_score": 10.0, "name": "Crit A"},
    2: {"score": 8.0, "weight": 50.0, "max_score": 10.0, "name": "Crit B"},
}

edge_suppliers = [
    ranker.SupplierScorecard("Zeta Corp",  "2026-01-05", 4.0, edge_case_scores),  # same PPI, same date, same exp -> loses on name
    ranker.SupplierScorecard("Alpha Corp", "2026-01-05", 4.0, edge_case_scores),  # same PPI, same date, same exp -> wins on name
    ranker.SupplierScorecard("Beta Corp",  "2026-01-03", 3.0, edge_case_scores),  # same PPI, earlier date -> wins overall
    ranker.SupplierScorecard("Gamma Corp", "2026-01-05", 5.0, edge_case_scores),  # same PPI, same date, higher exp -> 2nd overall
]

ranked_edge = ranker.rank_suppliers(edge_suppliers)
print(f"{'Rank':<5}{'Supplier':<12}{'PPI':<8}{'SubDate':<12}{'Exp':<5}  Trace")
for r in ranked_edge:
    print(f"{r.final_rank:<5}{r.supplier_name:<12}{r.ppi:<8}{r.submission_date:<12}{r.experience_rating:<5}  " + " | ".join(r.tie_break_trace))

assert [r.supplier_name for r in ranked_edge] == ["Beta Corp", "Gamma Corp", "Alpha Corp", "Zeta Corp"], (
    "Tie-break order did not match the expected PPI -> date -> experience -> name chain!"
)
print("\nTie-break order verified: Beta Corp -> Gamma Corp -> Alpha Corp -> Zeta Corp")


Rank Supplier    PPI     SubDate     Exp    Trace
1    Beta Corp   100.0   2026-01-03  3.0    PPI=100.00
2    Gamma Corp  100.0   2026-01-05  5.0    PPI=100.00 | tied on PPI with 'Beta Corp' -> compared submission_date
3    Alpha Corp  100.0   2026-01-05  4.0    PPI=100.00 | tied on PPI with 'Gamma Corp' -> compared submission_date | tied on submission_date -> compared experience_rating
4    Zeta Corp   100.0   2026-01-05  4.0    PPI=100.00 | tied on PPI with 'Alpha Corp' -> compared submission_date | tied on submission_date -> compared experience_rating | tied on experience_rating -> compared supplier name alphabetically

Tie-break order verified: Beta Corp -> Gamma Corp -> Alpha Corp -> Zeta Corp


## Cell 7 — End-to-end batch evaluation run test

Runs the FULL pipeline — Document Tool → Evaluation Agent (mock) →
Validation Tool → Ranking Tool → SQLite persistence — across all four
synthetic suppliers, exactly as the Streamlit "Evaluate batch" button will.


In [7]:
METADATA = {
    "Apex Systems":     {"submission_date": "2026-03-01", "experience_rating": 4.8},
    "BrightPath Tech":  {"submission_date": "2026-03-02", "experience_rating": 3.5},
    "NexaWorks":        {"submission_date": "2026-03-01", "experience_rating": 4.5},
    "Orbit Digital":    {"submission_date": "2026-03-03", "experience_rating": 4.2},
}

rfp_run_id = database.create_run(status="in_progress")
print(f"Created run: {rfp_run_id}\n")

scorecards = []
for supplier, meta in METADATA.items():
    proposal_text = extractions[supplier]["text"]
    raw = evaluator.evaluate_supplier(supplier, proposal_text, active_criteria, provider="mock")
    validated = validator.validate_llm_result(raw, active_criteria, expected_supplier_name=supplier)
    criteria_scores = {
        c.criterion_id: {"score": c.score, "weight": c.weight, "max_score": c.max_score, "name": c.name}
        for c in validated.criteria
    }
    scorecards.append(
        ranker.SupplierScorecard(
            supplier_name=supplier,
            submission_date=meta["submission_date"],
            experience_rating=meta["experience_rating"],
            criteria_scores=criteria_scores,
            warnings=validated.warnings,
            risks=validated.risks,
            overall_summary=validated.overall_summary,
        )
    )

ranked = ranker.rank_suppliers(scorecards)

print(f"{'Rank':<5}{'Supplier':<18}{'AbsScore':<10}{'PPI':<10}{'SubDate':<12}{'Exp':<5}")
for r in ranked:
    print(f"{r.final_rank:<5}{r.supplier_name:<18}{r.absolute_score:<10}{r.ppi:<10}{r.submission_date:<12}{r.experience_rating:<5}")
    database.save_supplier_result(
        rfp_run_id=rfp_run_id,
        supplier_name=r.supplier_name,
        submission_date=r.submission_date,
        experience_rating=r.experience_rating,
        absolute_score=r.absolute_score,
        ppi=r.ppi,
        final_rank=r.final_rank,
        result_payload={
            "criterion_breakdown": r.criterion_breakdown,
            "tie_break_trace": r.tie_break_trace,
            "warnings": r.warnings,
            "risks": r.risks,
            "overall_summary": r.overall_summary,
        },
    )

database.update_run_status(rfp_run_id, "completed")

persisted = database.get_results_for_run(rfp_run_id)
assert len(persisted) == len(METADATA)
print(f"\nPersisted and re-read {len(persisted)} results for run '{rfp_run_id}' from SQLite. OK.")
print("\nThis run_id can now be selected in the Streamlit app's 'Run History & JSON Export' page.")


Created run: RFP-20260927-145202-e33348

Rank Supplier          AbsScore  PPI       SubDate     Exp  
1    Apex Systems      75.77     90.3211   2026-03-01  4.8  
2    NexaWorks         62.74     79.7396   2026-03-01  4.5  
3    BrightPath Tech   57.22     73.7922   2026-03-02  3.5  
4    Orbit Digital     54.07     67.3511   2026-03-03  4.2  

Persisted and re-read 4 results for run 'RFP-20260927-145202-e33348' from SQLite. OK.

This run_id can now be selected in the Streamlit app's 'Run History & JSON Export' page.


## Summary

Every stage of the pipeline has been exercised in isolation and end-to-end,
with the strict **LLM ↔ math separation** preserved throughout: the mock (or
real) LLM only ever produced qualitative scores, justifications, and
evidence — every absolute score, benchmark, gap, relative %, PPI, and final
rank came from pure, deterministic Python in `ranker.py`.

Next step: `streamlit run app.py` to use the same modules through the full
5-page UI (Criteria Management, Supplier Input & Evaluate, Leaderboard,
Detailed Scorecard, Run History & JSON Export).
